# SQL Agent — build & test, node by node

Pipeline: `intent_classifier → injection_guard → schema injected → sql_generator → sql_validator → sql_optimizer → (re-validate / fallback) → (optional) executor → explainer`.

We build each node independently, test against Chinook, then wire them with LangGraph.

In [1]:
%pip install -q openai groq google-generativeai python-dotenv sqlglot langgraph langchain langchain-core pydantic


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from app.schema import CHINOOK_DDL, TABLES, RELATIONSHIPS
from app.llm_router import LLMRouter, Tier

print('tables:', list(TABLES.keys()))
print('relationships:', len(RELATIONSHIPS))
router = LLMRouter()
print('STRONG cohorts configured:', router.available_cohorts(Tier.STRONG))
print('CHEAP cohorts configured:', router.available_cohorts(Tier.CHEAP))

tables: ['Artist', 'Album', 'Genre', 'MediaType', 'Track', 'Employee', 'Customer', 'Invoice', 'InvoiceLine', 'Playlist', 'PlaylistTrack']
relationships: 11
STRONG cohorts configured: ['gemini', 'groq', 'openrouter']
CHEAP cohorts configured: ['gemini', 'groq', 'openrouter']


In [3]:
# Smoke test: one-shot ping to each tier (needs .env filled in)
print('STRONG:', router.chat(Tier.STRONG, user='Reply with the single word: pong').strip())
print('CHEAP :', router.chat(Tier.CHEAP,  user='Reply with the single word: pong').strip())

/Users/bhaskardev/Desktop/Reddit Chat Automation project/app/llm_router.py:42: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


STRONG: pong
CHEAP : pong


In [ ]:
# Confirm the DB works and the FKs are what we expect
import sqlite3, pandas as pd
DB = os.path.abspath('../data/Chinook_Sqlite.sqlite')
con = sqlite3.connect(DB)
pd.read_sql('SELECT COUNT(*) AS employees FROM Employee', con)

## Node 1 — intent classifier (+ scope)

One CHEAP LLM call. Returns `{intent, scope_reason, needs_clarification}`.

Intents: `generate_sql | optimize_sql | debug_sql | explain_sql | followup | out_of_scope`.
Scope reasons (when `out_of_scope`): `destructive | non_sql | ambiguous`.

In [ ]:
# TODO: fill in — intent_classifier(user_input, last_sql, messages) -> dict

## Node 2 — injection guard

In [ ]:
# TODO: fill in — injection_guard(user_input) -> bool (True = safe)

## Node 3 — SQL generator (uses schema + last_sql for follow-ups)

In [ ]:
# TODO: fill in

## Node 4 — SQL validator (sqlglot AST walk)

Checks: read-only, tables & columns exist, joins use real FKs, parses.

In [ ]:
# TODO: fill in

## Node 5 — optimizer (single pass + re-validate + fallback)

In [ ]:
# TODO: fill in

## Node 6 — executor (optional; EXPLAIN QUERY PLAN for cost)

In [ ]:
# TODO: fill in

## Node 7 — explainer (reads shared state, returns 1–3 lines)

In [ ]:
# TODO: fill in

## Wire with LangGraph + run the test matrix

In [ ]:
# TODO: StateGraph, conditional edges by intent, test matrix driver